# Creating Choropleth Map of Median AQI by US Counties

Needed data prep:

- AQI data ("annual_aqi_by_county_2026.csv") has county and state names, but doesn't have the FIPS codes for the corresponding state and county
- We can get corresponding FIPS codes from "aqs_sites.csv" file
- Then merge these 2 data sets to get a data set containing the median AQI and the FIPS codes
- FIPS code and a version of FIPS code without leading zero are needed to use charting libraries like Altair, Plotly, and Folium 

In [1]:
import polars as pl

In [2]:
df_aqi_by_counties = pl.read_csv(
    "../data/annual_aqi_by_county_2026.csv",
    columns=["State","County","Median AQI"],
    schema={
        "State": pl.String,
        "County": pl.String,
        "Median AQI": pl.Int64
    },
    extra_columns="ignore"
)

In [3]:
df_aqi_by_counties.shape

(827, 3)

In [4]:
df_aqi_by_counties.head()

State,County,Median AQI
str,str,i64
"""Alabama""","""Baldwin""",39
"""Alabama""","""Clay""",40
"""Alabama""","""DeKalb""",43
"""Alabama""","""Etowah""",47
"""Alabama""","""Jefferson""",52


In [5]:
df_aqs_sites = pl.read_csv(
    "../data/aqs_sites.csv",
    columns=["State Code","County Code","State Name","County Name"],
    schema={
        "State Code": pl.String,
        "County Code": pl.String,
        "State Name": pl.String,
        "County Name": pl.String
    },
    extra_columns="ignore"
)

In [6]:
df_aqs_sites.head()

State Code,County Code,State Name,County Name
str,str,str,str
"""01""","""001""","""Alabama""","""Autauga"""
"""01""","""001""","""Alabama""","""Autauga"""
"""01""","""001""","""Alabama""","""Autauga"""
"""01""","""003""","""Alabama""","""Baldwin"""
"""01""","""003""","""Alabama""","""Baldwin"""


In [7]:
df_fips_to_county_state_names = (
    df_aqs_sites.unique()
    .sort(pl.col("State Code"),pl.col("County Code"))
)

In [8]:
df_fips_to_county_state_names.head()

State Code,County Code,State Name,County Name
str,str,str,str
"""01""","""001""","""Alabama""","""Autauga"""
"""01""","""003""","""Alabama""","""Baldwin"""
"""01""","""005""","""Alabama""","""Barbour"""
"""01""","""007""","""Alabama""","""Bibb"""
"""01""","""009""","""Alabama""","""Blount"""


#### Join the 2 data sets so that the FIPS code and median AQI are all in one data set (called df_joined)

In [9]:
df_joined = df_aqi_by_counties.join(
    df_fips_to_county_state_names,
    how="left",
    left_on=["State","County"],
    right_on=["State Name","County Name"],
).sort(pl.col("State Code","County Code"))

In [10]:
df_joined.head()

State,County,Median AQI,State Code,County Code
str,str,i64,str,str
"""Alabama""","""Baldwin""",39,"""01""","""003"""
"""Alabama""","""Clay""",40,"""01""","""027"""
"""Alabama""","""DeKalb""",43,"""01""","""049"""
"""Alabama""","""Etowah""",47,"""01""","""055"""
"""Alabama""","""Jefferson""",52,"""01""","""073"""


#### Let's check that our joined or merged data set has the same number of rows as our original data set with median AQI values

In [11]:
assert df_joined.shape[0] == df_aqi_by_counties.shape[0]

#### Now create 5-digit FIPS code and id (5-digit FIPS without leading zero)

In [12]:
df_joined = (
    df_joined
    .with_columns(
        fips=pl.col("State Code") + pl.col("County Code"),  # Create FIPS code so that we can use Plotly
        id=(pl.col("State Code") + pl.col("County Code")).cast(pl.Int64)   # Create id (FIPS without leading zero) so that we can use Altair
    )
)

In [13]:
df_joined.head()

State,County,Median AQI,State Code,County Code,fips,id
str,str,i64,str,str,str,i64
"""Alabama""","""Baldwin""",39,"""01""","""003""","""01003""",1003
"""Alabama""","""Clay""",40,"""01""","""027""","""01027""",1027
"""Alabama""","""DeKalb""",43,"""01""","""049""","""01049""",1049
"""Alabama""","""Etowah""",47,"""01""","""055""","""01055""",1055
"""Alabama""","""Jefferson""",52,"""01""","""073""","""01073""",1073


## Creating Choropleth Map Using Plotly

In [ ]:
import json
import urllib.request
import plotly.express as px

# County boundaries keyed by 5-digit FIPS
url = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
with urllib.request.urlopen(url) as f:
    counties = json.load(f)

fig = px.choropleth(
    df_joined,
    geojson=counties,
    # locations="id",
    locations="fips",
    color="Median AQI",
    color_continuous_scale="Viridis",
    # range_color=(0, 15),                # clip outliers so colors stay readable
    scope="usa",                          # Albers USA projection, AK/HI insets
    hover_name="State",
    hover_data={"County": True, "Median AQI": ":d"},
    # labels={"Unemploy-ment Rate (%)": "% Unemployed"},
    title="2026 Median AQI by US Counties",
)

fig.update_traces(marker_line_width=0.1)   # thin county borders
fig.update_layout(
    margin=dict(l=0, r=0, t=40, b=0),
    width=1200,
    height=600
)
fig.show()

From above, we see that counties with no data do not get rendered in the map.  Below is modified version that adds a background (all counties rendered) with the foreground added containing counties with data rendered.

In [ ]:
import json
import urllib.request
import plotly.express as px
import plotly.graph_objects as go

url = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
with urllib.request.urlopen(url) as f:
    counties = json.load(f)

# Every county FIPS in the geojson
all_fips = [feat["id"] for feat in counties["features"]]

fig = px.choropleth(
    df_joined,
    geojson=counties,
    locations="fips",
    color="Median AQI",
    color_continuous_scale="Viridis",
    scope="usa",
    hover_name="State",
    hover_data={"County": True, "Median AQI": ":d"},
    title="2026 Median AQI by US Counties",
)

# Background layer: all counties, flat gray, no hover
background = go.Choropleth(
    geojson=counties,
    locations=all_fips,
    z=[0] * len(all_fips),
    colorscale=[[0, "#e6e6e6"], [1, "#e6e6e6"]],
    showscale=False,
    marker_line_width=0.1,
    marker_line_color="black",
    hoverinfo="skip",
)

# Insert background first so the data layer draws on top
fig.add_trace(background)
# Trace order matters. Plotly draws traces in order, so the reordering line puts the gray layer underneath.
# Without it, the gray counties would cover your data.
fig.data = (fig.data[-1],) + fig.data[:-1]

fig.update_traces(marker_line_width=0.1, selector=dict(type="choropleth"))
fig.update_layout(
    margin=dict(l=0, r=0, t=40, b=0),
    width=1200,
    height=600,
)
fig.show()

## Creating Choropleth Map Using Altair

In [ ]:
import altair as alt
from altair.datasets import data

counties = alt.topo_feature(data.us_10m.url, "counties")

chart = (
    alt.Chart(counties)
    .mark_geoshape()
    .encode(
        color=alt.Color(
            "Median AQI:Q",
            title="Median AQI",
            scale=alt.Scale(
                scheme="viridis",
                # reverse=True,
                # domain=[1,15],
                # clamp=True
            ),
        ),
        tooltip=[
            alt.Tooltip("County:N", title="County"),
            alt.Tooltip("State:N", title="State"),
            alt.Tooltip("Median AQI:Q", title="Median AQI", format=".0f"),
        ],
    )
    .transform_lookup(
        lookup="id",
        from_=alt.LookupData(
            df_joined,
            "id",
            ["Median AQI", "County", "State"],
        ),
    )
    .project(type="albersUsa")
    .properties(width=800, height=500, title="2026 Median AQI of US Counties")
)

chart

From above, we see counties with no data are not rendered at all because of the transform_lookup() method is essentially doing an INNER JOIN for rendering: when the lookup finds no match, the fields come back null, and Vega-Lite drops marks whose encoded value is null or invalid. The fix is to layer a background map of all counties underneath, so every county outline is always drawn, then draw the data layer on top.

In [ ]:
import altair as alt
from altair.datasets import data

counties = alt.topo_feature(data.us_10m.url, "counties")

# Layer 1: every county, light gray fill + borders (no data needed)
background = (
    alt.Chart(counties)
    .mark_geoshape(fill="#e6e6e6", stroke="white", strokeWidth=0.3)
    .project(type="albersUsa")
)

# Layer 2: counties with data, colored by Median AQI
foreground = (
    alt.Chart(counties)
    .mark_geoshape(stroke="white", strokeWidth=0.3)
    .encode(
        color=alt.Color(
            "Median AQI:Q",
            title="Median AQI",
            scale=alt.Scale(scheme="viridis"),
        ),
        tooltip=[
            alt.Tooltip("County:N", title="County"),
            alt.Tooltip("State:N", title="State"),
            alt.Tooltip("Median AQI:Q", title="Median AQI", format=".0f"),
        ],
    )
    .transform_lookup(
        lookup="id",
        from_=alt.LookupData(df_joined, "id", ["Median AQI", "County", "State"]),
    )
    .project(type="albersUsa")
)

chart = (
    (background + foreground)
    .properties(width=800, height=500, title="2026 Median AQI of US Counties")
)

chart

## Creating Choropleth Map Using Folium

Unfortunately, when using polars, can't use the convenience function: folium.Choropleth(), which accepts pandas dataframe, not polars dataframe.  Workaround is to use GeoJson() method and a style method if hover tooltip is desired.  Therefore, Folium API is more verbose compared to Altair or Plotly for county-level choropleth map creation.

In [ ]:
import folium
import polars as pl
import requests
from branca.colormap import linear

geo_url = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
counties = requests.get(geo_url).json()

# FIPS -> value lookup, create a new attribue or key called "median_aqi" in the features dict and assign median AQI value to each id (or FIPS code) 
# State code -> state name lookup, create a new attribute called "STATE_NAME" in the features dict and assign the state name to each STATE FIPS code
median_aqi = dict(df_joined.select("fips", "Median AQI").iter_rows())
state_codes = dict(df_joined.select("State Code", "State").iter_rows())
for feature in counties["features"]:
    feature["properties"]["median_aqi"] = median_aqi.get(feature["id"])  # None if missing
    feature["properties"]["STATE_NAME"] = state_codes.get(feature["properties"]["STATE"])

# Color scale from the data range
colormap = linear.YlOrRd_09.scale(df_joined["Median AQI"].min(), df_joined["Median AQI"].max())
colormap.caption = "Median API"

def style_fn(feature):
    val = feature["properties"]["median_aqi"]
    return {
        "fillColor": colormap(val) if val is not None else "lightgray",
        "fillOpacity": 0.8,
        "color": "black",
        "weight": 0.2,
    }

def highlight_fn(feature):
    return {"weight": 2, "color": "black", "fillOpacity": 0.95}

m = folium.Map(location=[38, -96], zoom_start=4, tiles=None)

folium.GeoJson(
    counties,
    style_function=style_fn,
    highlight_function=highlight_fn,
    tooltip=folium.GeoJsonTooltip(
        fields=["STATE_NAME", "NAME", "median_aqi"],
        aliases=["State:", "County:", "Median AQI:"],
        localize=True,
    ),
).add_to(m)

colormap.add_to(m)  # legend
m

## Creating Choropleth Map Using GeoLibre

In [ ]:
import json
import urllib.request
from geolibre import Map

url = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
with urllib.request.urlopen(url) as f:
    counties = json.load(f)

# Lookup of your data by FIPS (polars; for pandas use df_joined.to_dict("records"))
lookup = {
    str(r["fips"]).zfill(5): r
    for r in df_joined.iter_rows(named=True)
}

# Data layer: only counties that have a Median AQI, with values copied onto properties
data_features = []
for feat in counties["features"]:
    row = lookup.get(feat["id"])
    if row is None or row["Median AQI"] is None:
        continue
    data_features.append({
        "type": "Feature",
        "id": feat["id"],
        "geometry": feat["geometry"],
        "properties": {
            "fips": feat["id"],
            "State": row["State"],
            "County": row["County"],
            "median_aqi": int(round(row["Median AQI"])),
        },
    })
counties_aqi = {"type": "FeatureCollection", "features": data_features}

m = Map(center=(-98.6, 39.8), zoom=3, height="600px")
m.set_projection("mercator")   # flat map instead of the default globe

# Background layer: all counties, flat gray, no popup or hover. Added first, so it draws underneath.
m.add_geojson(
    counties,
    name="All counties",
    fillColor="#e6e6e6",
    strokeColor="#000000",
    strokeWidth=0.1,
    popup=False,
    tooltip=False,
)

# Data layer on top
m.add_choropleth(
    counties_aqi,
    column="median_aqi",
    name="2026 Median AQI by US Counties",
    colormap="viridis",
    scheme="quantile",
    class_count=7,
    strokeWidth=0.1,
    popup={
        "title": "County",
        "fields": [
            {"field": "County", "label": "County"},
            {"field": "State", "label": "State"},
            {"field": "median_aqi", "label": "Median AQI", "kind": "number", "decimals": 0},
        ],
    },
    tooltip="County",
)

m.set_map_legend("Median AQI", position="bottom-right")
m.set_identify("2026 Median AQI by US Counties")   # popups only on the data layer

m

## Plotting Locations of Monitoring Sites with Plotly

In [14]:
df_aqs_sites = pl.read_csv(
    "../data/aqs_sites.csv",
    columns=["Site Number","Site Established Date","Site Closed Date","Latitude","Longitude","City Name","State Name"],
    schema_overrides={
        "Latitude": pl.Float64,
        "Longitude": pl.Float64
    },
    infer_schema_length=10000,
)

In [15]:
df_aqs_sites.head()

Site Number,Site Established Date,Site Closed Date,Latitude,Longitude,City Name,State Name
i64,str,str,f64,f64,str,str
1,"""1974-05-01""","""1976-12-31""",32.437458,-86.472891,"""Prattville""","""Alabama"""
2,"""1980-01-01""","""1982-12-31""",32.42847,-86.443585,"""Prattville""","""Alabama"""
3,"""1989-08-31""","""1990-11-30""",32.332659,-86.791521,"""Not in a City""","""Alabama"""
1,"""1959-01-01""","""1959-12-31""",0.0,0.0,"""Not in a city""","""Alabama"""
2,"""1974-01-01""","""1977-12-31""",30.552367,-87.706911,"""Robertsdale""","""Alabama"""


#### Filter our data to active (not closed) sites and only sites located within continental United States, Alaska, and Hawaii

In [16]:
df = (
    df_aqs_sites
    # Filter our data to active (not closed) sites and only sites located within continental United States, Alaska, and Hawaii
    .filter(
        (pl.col("Site Closed Date") == "") &
        (pl.col("State Name") != "Country Of Mexico") &
        (pl.col("State Name") != "Guam") &
        (pl.col("State Name") != "Canada") &
        (pl.col("State Name") != "Puerto Rico") &
        (pl.col("State Name") != "Virgin Islands") &
        (pl.col("Latitude").is_not_null()) &
        (pl.col("Longitude").is_not_null()) &
        ((pl.col("Latitude") != 0) & (pl.col("Longitude") != 0))
    )
)

In [ ]:
import plotly.express as px
import polars as pl

df_aqs_sites = pl.read_csv(
    "../data/aqs_sites.csv",
    columns=["Site Number","Site Established Date","Site Closed Date","Latitude","Longitude","City Name","State Name"],
    schema_overrides={
        "Latitude": pl.Float64,
        "Longitude": pl.Float64
    },
    infer_schema_length=10000,
)

df = (
    df_aqs_sites
    # Filter our data to active (not closed) sites and only sites located within continental United States, Alaska, and Hawaii
    .filter(
        (pl.col("Site Closed Date") == "") &
        (pl.col("State Name") != "Country Of Mexico") &
        (pl.col("State Name") != "Guam") &
        (pl.col("State Name") != "Canada") &
        (pl.col("State Name") != "Puerto Rico") &
        (pl.col("State Name") != "Virgin Islands") &
        (pl.col("Latitude").is_not_null()) &
        (pl.col("Longitude").is_not_null()) &
        (pl.col("Latitude") != 0) & (pl.col("Longitude") != 0)
    )
)

fig = px.scatter_map(
    df, 
    lat="Latitude", 
    lon="Longitude", 
    hover_data=["City Name","State Name"],
    zoom=3,
    center={"lat": 39.8, "lon": -98.6},
    map_style="open-street-map",
)

fig.update_layout(
    margin=dict(l=0, r=0, t=40, b=0),
    width=1200,
    height=600,
)

fig.show()

## Plotting Locations of Monitoring Sites with Altair

In [ ]:
import altair as alt
import polars as pl
from vega_datasets import data

df_aqs_sites = pl.read_csv(
    "../data/aqs_sites.csv",
    columns=["Site Number","Site Established Date","Site Closed Date","Latitude","Longitude","City Name","State Name"],
    schema_overrides={
        "Latitude": pl.Float64,
        "Longitude": pl.Float64
    },
    infer_schema_length=10000,
)

df = (
    df_aqs_sites
    # Filter our data to active (not closed) sites and only sites located within continental United States, Alaska, and Hawaii
    .filter(
        (pl.col("Site Closed Date") == "") &
        (pl.col("State Name") != "Country Of Mexico") &
        (pl.col("State Name") != "Guam") &
        (pl.col("State Name") != "Canada") &
        (pl.col("State Name") != "Puerto Rico") &
        (pl.col("State Name") != "Virgin Islands") &
        (pl.col("Latitude").is_not_null()) &
        (pl.col("Longitude").is_not_null()) &
        ((pl.col("Latitude") != 0) & (pl.col("Longitude") != 0))
    )
)

states = alt.topo_feature(data.us_10m.url, "states")

background = alt.Chart(states).mark_geoshape(
    fill="lightgray", stroke="white"
).project("albersUsa").properties(width=800, height=600)

points = alt.Chart(df).mark_circle(size=10, color="red").encode(
    longitude="Longitude:Q",
    latitude="Latitude:Q",
    tooltip=["City Name:N", "State Name:N"],
).project("albersUsa")

background + points

Without being able to zoom in, using Altair for plotting of several markers is not very useful.

## Plotting Locations of Monitoring Sites with Folium

In [ ]:
import folium
import polars as pl

df_aqs_sites = pl.read_csv(
    "../data/aqs_sites.csv",
    columns=["Site Number","Site Established Date","Site Closed Date","Latitude","Longitude","City Name","State Name"],
    schema_overrides={
        "Latitude": pl.Float64,
        "Longitude": pl.Float64
    },
    infer_schema_length=10000,
)

df = (
    df_aqs_sites
    # Filter our data to active (not closed) sites and only sites located within continental United States, Alaska, and Hawaii
    .filter(
        (pl.col("Site Closed Date") == "") &
        (pl.col("State Name") != "Country Of Mexico") &
        (pl.col("State Name") != "Guam") &
        (pl.col("State Name") != "Canada") &
        (pl.col("State Name") != "Puerto Rico") &
        (pl.col("State Name") != "Virgin Islands") &
        (pl.col("Latitude").is_not_null()) &
        (pl.col("Longitude").is_not_null()) &
        ((pl.col("Latitude") != 0) & (pl.col("Longitude") != 0))
    )
)

# Centered on the continental US
m = folium.Map(location=[39.8, -98.6], zoom_start=4, tiles="OpenStreetMap")

for row in df.iter_rows(named=True):
    folium.Marker(
        location=[row["Latitude"], row["Longitude"]],
        tooltip=row["City Name"],
        popup=row["City Name"],
    ).add_to(m)

m  # in Jupyter, this line displays the map inline

From above, we can see that because Folium's markers are rather large, it is not visually informative to plot or place several individual markers on the map.  Instead, we can use Folium's marker group features like FeatureGroup and MarkerCluser.  Below is the modified code using those 2 features:

In [ ]:
import folium
from folium.plugins import MarkerCluster
import polars as pl

df_aqs_sites = pl.read_csv(
    "../data/aqs_sites.csv",
    columns=["Site Number","Site Established Date","Site Closed Date","Latitude","Longitude","City Name","State Name"],
    schema_overrides={
        "Latitude": pl.Float64,
        "Longitude": pl.Float64
    },
    infer_schema_length=10000,
)

df = (
    df_aqs_sites
    # Filter our data to active (not closed) sites and only sites located within continental United States, Alaska, and Hawaii
    .filter(
        (pl.col("Site Closed Date") == "") &
        (pl.col("State Name") != "Country Of Mexico") &
        (pl.col("State Name") != "Guam") &
        (pl.col("State Name") != "Canada") &
        (pl.col("State Name") != "Puerto Rico") &
        (pl.col("State Name") != "Virgin Islands") &
        (pl.col("Latitude").is_not_null()) &
        (pl.col("Longitude").is_not_null()) &
        ((pl.col("Latitude") != 0) & (pl.col("Longitude") != 0))
    )
)

# Centered on the continental US
m = folium.Map(location=[39.8, -98.6], zoom_start=4, tiles="OpenStreetMap")

# One toggleable FeatureGroup per state, each containing a MarkerCluster
for state in df["State Name"].unique().sort().to_list():
    state_df = df.filter(pl.col("State Name") == state)

    fg = folium.FeatureGroup(name=f"{state} ({state_df.height})", show=True)
    cluster = MarkerCluster().add_to(fg)

    for row in state_df.iter_rows(named=True):
        city = row["City Name"] or "Unknown"
        folium.Marker(
            location=[row["Latitude"], row["Longitude"]],
            tooltip=city,
            popup=city,
        ).add_to(cluster)

    fg.add_to(m)

folium.LayerControl(collapsed=True).add_to(m)

m  # in Jupyter, this line displays the map inline

Now this version has useful clustering of markers and past a certain zoom level, we can see individual markers.

## Plotting Locations of Monitoring Sites with GeoLibre

In [ ]:
from geolibre import Map
import polars as pl

df_aqs_sites = pl.read_csv(
    "../data/aqs_sites.csv",
    columns=["Site Number","Site Established Date","Site Closed Date","Latitude","Longitude","City Name","State Name"],
    schema_overrides={
        "Latitude": pl.Float64,
        "Longitude": pl.Float64
    },
    infer_schema_length=10000,
)

df = (
    df_aqs_sites
    # Filter our data to active (not closed) sites and only sites located within continental United States, Alaska, and Hawaii
    .filter(
        (pl.col("Site Closed Date") == "") &
        (pl.col("State Name") != "Country Of Mexico") &
        (pl.col("State Name") != "Guam") &
        (pl.col("State Name") != "Canada") &
        (pl.col("State Name") != "Puerto Rico") &
        (pl.col("State Name") != "Virgin Islands") &
        (pl.col("Latitude").is_not_null()) &
        (pl.col("Longitude").is_not_null()) &
        ((pl.col("Latitude") != 0) & (pl.col("Longitude") != 0))
    )
)

# Centered on the continental US. NOTE: GeoLibre uses (lng, lat), not (lat, lng).
m = Map(center=(-98.6, 39.8), zoom=4)
m.add_tile_layer(
    "https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap",
    attribution="(c) OpenStreetMap contributors",
)

# One toggleable clustered layer per state
for state in df["State Name"].unique().sort().to_list():
    state_df = df.filter(pl.col("State Name") == state)

    points = [
        {
            "lng": row["Longitude"],
            "lat": row["Latitude"],
            "city": row["City Name"] or "Unknown",
        }
        for row in state_df.iter_rows(named=True)
    ]

    m.add_marker_cluster(
        points,
        name=f"{state} ({state_df.height})",
        popup="city",     # click popup shows the city
        tooltip="city",   # hover tooltip shows the city
    )

# Popups only open while the Identify tool is armed
m.set_identify()

m  # in Jupyter, this line displays the map inline